# Five questions with local vLLM (Metal or CUDA)

Start `./scripts/start-mlx-vllm.sh` on Apple Silicon or `./scripts/start-cuda-vllm.sh` on Linux CUDA and leave that terminal open. See [README setup](../README.md) first.

Select **DIY-AI workshop (Python 3.12)**, edit the configuration below, and run all cells in order. The default model matches the host's launcher; override `VLLM_MODEL`, `VLLM_BASE_URL`, and optionally `VLLM_API_KEY` before starting Jupyter, or edit the setup cell. Discovery checks the server's actual model IDs.

This notebook sends requests only to the configured vLLM server. It runs the workshop's five questions sequentially, after one warm-up request. The original 150-token budget is deliberately small and can cut off answers; inspect `finish_reason` and raise `MAX_TOKENS` to experiment. This is a responsiveness/completeness smoke test, not a correctness score.

Historical comparisons remain in the presentation. Run the separate Spark notebook if you have workshop VPN access.


In [ ]:
import os
import platform
import sys
import time
from collections import Counter
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display

PROJECT_ROOT = next(
    directory
    for directory in (Path.cwd(), *Path.cwd().parents)
    if (directory / "pyproject.toml").is_file()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

BASE_URL = (
    os.environ.get("VLLM_BASE_URL", "http://localhost:8000")
    .rstrip("/")
    .removesuffix("/v1")
)

DEFAULT_MODEL = (
    "mlx-community/gemma-4-e4b-it-4bit"
    if platform.system() == "Darwin" and platform.machine() == "arm64"
    else "google/gemma-4-E4B-it"
)
MODEL_NAMES = [os.environ.get("VLLM_MODEL", DEFAULT_MODEL)]

# Add other installed/served model IDs to MODEL_NAMES to compare them.
API_KEY = os.environ.get("VLLM_API_KEY", "")
MAX_TOKENS = 150
TEMPERATURE = 0.7
NUM_REQUESTS = 5
REQUEST_TIMEOUT = (10, 180)  # Connection timeout, then inference timeout.
BENCHMARK_PROMPTS = [
    "What is the capital of France?",
    "Write a short poem about artificial intelligence.",
    "Explain quantum computing in simple terms.",
    "What are the three laws of robotics?",
    "Summarize the plot of the movie Inception.",
]
headers = {"Content-Type": "application/json"}
if API_KEY:
    headers["Authorization"] = f"Bearer {API_KEY}"
CHAT_URL = f"{BASE_URL}/v1/chat/completions"
print(f"Endpoint: {CHAT_URL}")
print(f"Models: {MODEL_NAMES}")

## Discover served models
Run this cell before testing; choose model IDs from its output.

In [ ]:
# List the actual model IDs advertised by the inference server.
response = requests.get(
    f"{BASE_URL}/v1/models",
    headers=headers,
    timeout=REQUEST_TIMEOUT,
)
response.raise_for_status()
available_models = [item["id"] for item in response.json()["data"]]
print("Available models:", available_models)
missing_models = set(MODEL_NAMES) - set(available_models)
if missing_models:
    raise ValueError(
        f"Models not served: {sorted(missing_models)}. "
        "Update MODEL_NAMES using the IDs listed above and rerun setup."
    )

## Request helper and smoke test

In [ ]:
def chat(prompt, model):
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": MAX_TOKENS,
        "temperature": TEMPERATURE,
        "stream": False,
    }
    start = time.perf_counter()
    response = requests.post(
        CHAT_URL,
        headers=headers,
        json=payload,
        timeout=REQUEST_TIMEOUT,
    )
    response.raise_for_status()
    data = response.json()
    elapsed = time.perf_counter() - start
    choices = data.get("choices")
    if not isinstance(choices, list) or not choices:
        raise ValueError("Response has no chat completion choices")
    choice = choices[0]
    message = choice.get("message") or {}
    text = message.get("content") or ""
    if not isinstance(text, str):
        raise TypeError("Expected string message.content")
    reasoning = message.get("reasoning_content") or message.get("reasoning") or ""
    if not text.strip() and not reasoning:
        raise ValueError("Model returned no text or reasoning")
    usage = data.get("usage") or {}
    completion_tokens = usage.get("completion_tokens")
    return {
        "model": model,
        "served_model": data.get("model", model),
        "latency_s": elapsed,
        "prompt_tokens": usage.get("prompt_tokens"),
        "completion_tokens": completion_tokens,
        "tokens/s": completion_tokens / elapsed
        if completion_tokens is not None
        else None,
        "finish_reason": choice.get("finish_reason"),
        "generated_text": text,
        "reasoning_text": reasoning,
        "status": "OK" if text.strip() else "NO_VISIBLE_TEXT",
        "error": "",
    }

In [ ]:
TEST_PROMPT = "Explain attention mechanisms in transformer models in 2-3 sentences."
for model in MODEL_NAMES:
    result = chat(TEST_PROMPT, model)
    print(f"\n{model}: {result['latency_s']:.3f}s ({result['finish_reason']})")
    print(result["generated_text"] or "No visible answer; increase MAX_TOKENS.")

## Benchmark
Full prompts and responses are retained. HTTP, decoding, and malformed-response failures are excluded from success statistics. Missing token usage remains unknown. Completion tokens per second includes end-to-end request overhead and any reasoning tokens reported by the server. `NO_VISIBLE_TEXT` indicates a reasoning-only response; increase the token budget if needed. A `length` finish reason indicates truncation.

In [ ]:
if NUM_REQUESTS < 1 or not BENCHMARK_PROMPTS or not MODEL_NAMES:
    raise ValueError("Configure at least one model, prompt, and request")

responses_collected = []
for model in MODEL_NAMES:
    for i in range(NUM_REQUESTS):
        prompt = BENCHMARK_PROMPTS[i % len(BENCHMARK_PROMPTS)]
        try:
            result = chat(prompt, model)
        except (requests.RequestException, ValueError, KeyError, TypeError) as error:
            result = {
                "model": model,
                "served_model": None,
                "latency_s": None,
                "prompt_tokens": None,
                "completion_tokens": None,
                "tokens/s": None,
                "finish_reason": None,
                "generated_text": "",
                "reasoning_text": "",
                "status": "FAILED",
                "error": str(error),
            }
        result.update(request=i + 1, prompt=prompt)
        responses_collected.append(result)
        print(f"{model} {i + 1}/{NUM_REQUESTS}: {result['status']}")

df_results = pd.DataFrame(responses_collected)
display(df_results)
summary_rows = []
for model in MODEL_NAMES:
    rows = df_results[df_results["model"] == model]
    ok = rows[rows["status"] == "OK"]
    # Only measured visible answers enter latency and throughput statistics.
    total_time = ok["latency_s"].sum()
    known_usage = not ok.empty and ok["completion_tokens"].notna().all()
    summary_rows.append(
        {
            "model": model,
            "successful": len(ok),
            "failed": int((rows["status"] == "FAILED").sum()),
            "no_visible_text": int((rows["status"] == "NO_VISIBLE_TEXT").sum()),
            "mean_latency_s": ok["latency_s"].mean(),
            "median_latency_s": ok["latency_s"].median(),
            "min_latency_s": ok["latency_s"].min(),
            "max_latency_s": ok["latency_s"].max(),
            "completion_tokens/s": (
                ok["completion_tokens"].sum() / total_time
                if known_usage and total_time > 0
                else None
            ),
        }
    )
df_summary = pd.DataFrame(summary_rows)
display(df_summary)

## Inspect response text
These descriptive length, vocabulary, and repetition measures do not evaluate factual correctness. They reuse the measured responses without issuing extra requests. Compare notebook runs only with matching prompts, token budgets, temperatures, and server conditions; no historical cluster numbers are inserted.

In [ ]:
def compute_quality_metrics(text):
    words = text.lower().split()
    trigrams = list(zip(words, words[1:], words[2:]))
    counts = Counter(trigrams)
    return {
        "char_len": len(text),
        "word_count": len(words),
        "type_token_ratio": len(set(words)) / len(words) if words else 0.0,
        "repeat_3gram_ratio": (
            sum(count - 1 for count in counts.values()) / len(trigrams)
            if trigrams
            else 0.0
        ),
    }


quality_rows = [
    {
        "model": row["model"],
        "request": row["request"],
        "prompt": row["prompt"],
        **compute_quality_metrics(row["generated_text"]),
    }
    for row in responses_collected
    if row["status"] == "OK"
]
df_quality = pd.DataFrame(quality_rows)
if df_quality.empty:
    print("No successful visible responses to inspect. See df_results for errors.")
else:
    display(df_quality)
    display(df_quality.groupby("model").mean(numeric_only=True).drop(columns="request"))
    for row in responses_collected:
        if row["status"] == "OK":
            print(f"\n[{row['model']}] {row['prompt']}")
            print(row["generated_text"])